In [ ]:
!pip install timm anthropic -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm
import pandas as pd
import numpy as np
import json
import os
import re
import random
import base64
import io
from PIL import Image, ImageEnhance
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')

BASE             = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER     = f'{BASE}/images_human_fixed'
IMAGE_FOLDER     = f'{BASE}/images'
AUG_FOLDER       = f'{BASE}/images_human_aug'
RESULT_FOLDER    = f'{BASE}/results'
NEW_FEATURE_PATH = f'{BASE}/labels/vqa_features_v2.json'
MODEL_FOLDER     = f'{BASE}/models_improved'

os.makedirs(MODEL_FOLDER,  exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
CUTOFF = 6

# โหลดข้อมูล
train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)
val_df   = pd.read_csv(
    f'{BASE}/labels/human_val.csv'
)
test_df  = pd.read_csv(
    f'{BASE}/labels/human_test.csv'
)
for d in [train_df, val_df, test_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

print(f'Device: {device}')
print(f'Train : {len(train_df)} รูป')
print(f'Val   : {len(val_df)} รูป')
print(f'Test  : {len(test_df)} รูป')

In [ ]:
import anthropic

from google.colab import userdata
client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

def encode_image(img_path):
    try:
        img    = Image.open(img_path).convert('RGB')
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=95)
        return base64.standard_b64encode(
            buffer.getvalue()
        ).decode('utf-8')
    except:
        return None

def get_img_path(filename):
    paths = [
        f'{HUMAN_FOLDER}/{filename}',
        f'{IMAGE_FOLDER}/{filename}',
        f'{IMAGE_FOLDER}/{filename.replace(".jpg",".tif")}',
        f'{HUMAN_FOLDER}/{filename.replace(".tif",".jpg")}',
    ]
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def get_vqa_feat(filename, vqa_dict):
    if filename in vqa_dict:
        return vqa_dict[filename]
    alt = filename.replace('.jpg','.tif') \
          if '.jpg' in filename \
          else filename.replace('.tif','.jpg')
    if alt in vqa_dict:
        return vqa_dict[alt]
    return {}

NEW_VQA_PROMPT = """
Analyze this Clock Drawing Test (CDT) image carefully.
The correct time to draw is 11:10.

IMPORTANT for reading clock hands:
- "11" is at top-LEFT of the clock
- "1"  is at top-RIGHT of the clock
- For 11:10: hour hand points LEFT of 12 (toward 11)
             minute hand points RIGHT of 12 (toward 2)

Return ONLY a JSON object with NO extra text:
{
  "digit_count": <int 0-12>,
  "missing_numbers": <list e.g. [3,7] or []>,
  "numbers_in_correct_position": <int 0-12>,
  "top_quadrant_count": <int 0-3>,
  "right_quadrant_count": <int 0-3>,
  "bottom_quadrant_count": <int 0-3>,
  "left_quadrant_count": <int 0-3>,
  "numbers_crowded": <"Yes"/"No">,
  "numbers_outside_circle": <"Yes"/"No">,
  "circle_complete": <"Yes"/"No">,
  "circle_shape": <"round"/"oval"/"irregular">,
  "numbers_evenly_spaced": <"Yes"/"No">,
  "largest_gap_position": <"top"/"right"/"bottom"/"left"/"none">,
  "hour_hand_exists": <"Yes"/"No">,
  "minute_hand_exists": <"Yes"/"No">,
  "hour_points_to": <int 1-12 or -1 if absent>,
  "minute_points_to": <int 1-12 or -1 if absent>,
  "hands_distinguishable": <"Yes"/"No">,
  "overall_quality": <int 1-5>,
  "digit_count_confidence": <int 1-5>,
  "hands_confidence": <int 1-5>,
  "spatial_confidence": <int 1-5>
}
"""

def fix_hand_features(features):
    """Rule-based fix hand features"""
    if not features or 'error' in features:
        return features

    h    = features.get('hour_points_to', -1)
    m    = features.get('minute_points_to', -1)
    conf = features.get('hands_confidence', 5)

    if conf <= 2:
        return features

    if h == 1 and m == 2:
        features['hour_points_to'] = 11

    elif h == 1 and 1 <= m <= 3:
        features['hour_points_to'] = 11

    return features

def extract_vqa_v2(img_path):
    """Extract VQA features v2"""
    img_b64 = encode_image(img_path)
    if img_b64 is None:
        return None
    try:
        response = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=600,
            messages=[{
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type'      : 'base64',
                            'media_type': 'image/jpeg',
                            'data'      : img_b64,
                        }
                    },
                    {
                        'type': 'text',
                        'text': NEW_VQA_PROMPT
                    }
                ]
            }]
        )
        text = response.content[0].text.strip()
        m    = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            text = m.group()
        features = json.loads(text)
        features = fix_hand_features(features)
        return features
    except:
        return None

In [ ]:
sample_files = test_df['filename'].tolist()[:5]

for filename in sample_files:
    src  = get_img_path(filename)
    feat = extract_vqa_v2(src)

    row    = test_df[test_df['filename']==filename]
    true_e = row['domain_E'].values[0]
    true_d = row['domain_D'].values[0]

    if feat:
        h    = feat.get('hour_points_to')
        m    = feat.get('minute_points_to')
        conf = feat.get('hands_confidence')
        print(f'{filename} (D={true_d} E={true_e}):')
        print(f'  digit_count : {feat.get("digit_count")}')
        print(f'  missing     : {feat.get("missing_numbers")}')
        print(f'  hour→{h} min→{m} conf={conf}')
        print(f'  circle_shape: {feat.get("circle_shape")}')
        print(f'  evenly_spaced: {feat.get("numbers_evenly_spaced")}')
    else:
        print(f'{filename}: ❌')
    print()

In [ ]:
all_df = pd.read_csv(
    f'{BASE}/labels/human_labels.csv'
)
print(f'รูปทั้งหมด: {len(all_df)}')

if os.path.exists(NEW_FEATURE_PATH):
    with open(NEW_FEATURE_PATH) as f:
        vqa_v2 = json.load(f)
    print(f'มี progress: {len(vqa_v2)} รูป')
else:
    vqa_v2 = {}

remaining = [
    row for _, row in all_df.iterrows()
    if row['filename'] not in vqa_v2
    or 'error' in vqa_v2.get(row['filename'], {})
]
print(f'ต้อง extract: {len(remaining)} รูป')
print(f'ประมาณ: {len(remaining)*2/60:.0f} นาที')

error_count = 0
for row in tqdm(remaining):
    filename = row['filename']
    src      = get_img_path(filename)

    if src is None:
        error_count += 1
        vqa_v2[filename] = {'error': True}
        continue

    feat = extract_vqa_v2(src)
    if feat:
        vqa_v2[filename] = feat
    else:
        error_count += 1
        vqa_v2[filename] = {'error': True}

    if len(vqa_v2) % 50 == 0:
        with open(NEW_FEATURE_PATH, 'w') as f:
            json.dump(vqa_v2, f)

with open(NEW_FEATURE_PATH, 'w') as f:
    json.dump(vqa_v2, f, indent=2)

success = sum(
    1 for v in vqa_v2.values()
    if 'error' not in v
)
print(f'\nสำเร็จ: {success} รูป')
print(f'Error   : {error_count} รูป')

In [ ]:
def features_to_vector_v2(features):
    """
    VQA features v2 — 23 dims
    """
    if features is None or 'error' in features:
        return np.zeros(23)

    vec = []

    # ── Digit Group (0-9) ช่วย A, B ──
    vec.append(
        features.get('digit_count', 0) / 12.0)
    missing = features.get('missing_numbers', [])
    vec.append(len(missing) / 12.0)
    vec.append(
        features.get(
            'numbers_in_correct_position', 0
        ) / 12.0)
    for q in ['top','right','bottom','left']:
        vec.append(
            min(features.get(
                f'{q}_quadrant_count', 0
            ) / 3.0, 1.0))
    vec.append(
        1.0 if features.get(
            'numbers_crowded') == 'Yes' else 0.0)
    vec.append(
        1.0 if features.get(
            'numbers_outside_circle') == 'Yes'
        else 0.0)

    # ── Spatial Group (10-14) ช่วย C ──
    vec.append(
        1.0 if features.get(
            'circle_complete') == 'Yes' else 0.0)
    shape     = features.get('circle_shape','round')
    shape_map = {
        'round'    : [1,0,0],
        'oval'     : [0,1,0],
        'irregular': [0,0,1],
    }
    vec.extend(shape_map.get(shape, [1,0,0]))
    vec.append(
        1.0 if features.get(
            'numbers_evenly_spaced') == 'Yes'
        else 0.0)

    # ── Hand Group (15-19) ช่วย D, E ──
    vec.append(
        1.0 if features.get(
            'hour_hand_exists') == 'Yes' else 0.0)
    vec.append(
        1.0 if features.get(
            'minute_hand_exists') == 'Yes' else 0.0)
    h_pos = features.get('hour_points_to', -1)
    vec.append(h_pos/12.0 if h_pos > 0 else 0.0)
    m_pos = features.get('minute_points_to', -1)
    vec.append(m_pos/12.0 if m_pos > 0 else 0.0)
    vec.append(
        1.0 if features.get(
            'hands_distinguishable') == 'Yes'
        else 0.0)

    # ── Quality + Confidence (20-22) ──
    vec.append(
        features.get('overall_quality', 1) / 5.0)
    vec.append(
        features.get(
            'digit_count_confidence', 3) / 5.0)
    vec.append(
        features.get(
            'hands_confidence', 3) / 5.0)
    vec.append(
        features.get(
            'spatial_confidence', 3) / 5.0)

    return np.array(vec, dtype=np.float32)

if vqa_v2:
    sample     = next(
        v for v in vqa_v2.values()
        if 'error' not in v
    )
    vec        = features_to_vector_v2(sample)
    VQA_DIM_V2 = len(vec)
    print(f'Feature vector v2: {VQA_DIM_V2} dims')
    print(f'  Digit   (0-9) : {vec[:10]}')
    print(f'  Spatial(10-14): {vec[10:15]}')
    print(f'  Hand   (15-19): {vec[15:20]}')
    print(f'  Quality(20-22): {vec[20:]}')
else:
    VQA_DIM_V2 = 23
    print(f'VQA_DIM_V2 = {VQA_DIM_V2}')

In [ ]:
class CDTDataset(Dataset):
    def __init__(self, df, image_folder,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB',(224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, labels

class CDTHybridDatasetV2(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, aug_folder=None,
                 transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB',(224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        feat_dict = get_vqa_feat(filename, self.vqa_feats)
        feat_vec  = torch.tensor(
            features_to_vector_v2(feat_dict),
            dtype=torch.float32
        )
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, feat_vec, labels

DIGIT_DIM   = 10
SPATIAL_DIM = 5
HAND_DIM    = 5
QUALITY_DIM = 3

class DomainSpecificEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.digit_enc = nn.Sequential(
            nn.Linear(DIGIT_DIM, 32),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU()
        )
        self.spatial_enc = nn.Sequential(
            nn.Linear(SPATIAL_DIM, 16),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 8), nn.ReLU()
        )
        self.hand_enc = nn.Sequential(
            nn.Linear(HAND_DIM, 16),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 8), nn.ReLU()
        )
        self.quality_enc = nn.Sequential(
            nn.Linear(QUALITY_DIM, 8),
            nn.ReLU()
        )
        self.output_dim = 16 + 8 + 8 + 8  # = 40

    def forward(self, x):
        digit   = self.digit_enc(x[:, :10])
        spatial = self.spatial_enc(x[:, 10:15])
        hand    = self.hand_enc(x[:, 15:20])
        quality = self.quality_enc(x[:, 20:])
        return torch.cat(
            [digit, spatial, hand, quality], dim=1
        )

class CDTHybridModelV2(nn.Module):
    def __init__(self, backbone_name='vit',
                 num_domains=5):
        super().__init__()

        if backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True, num_classes=0
            )
            vis_dim = 768
        else:
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
            vis_dim = 2048

        self.backbone_name = backbone_name
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.vis_shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(vis_dim, 512),
            nn.ReLU(), nn.Dropout(0.3)
        )

        self.vqa_encoder = DomainSpecificEncoder()
        vqa_out_dim      = self.vqa_encoder.output_dim

        fused_dim = 512 + vqa_out_dim
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(), nn.Dropout(0.2)
        )

        self.heads_reg = nn.ModuleList([
            nn.Linear(128, 1)
            for _ in range(num_domains)
        ])
        self.heads_cls = nn.ModuleList([
            nn.Linear(128, 3)
            for _ in range(num_domains)
        ])

    def forward(self, img, vqa_feat, mode='reg'):
        if self.backbone_name == 'vit':
            vis = self.backbone(img)
            vis = vis.unsqueeze(-1).unsqueeze(-1)
        else:
            vis = self.backbone(img)
        vis   = self.vis_shared(vis)
        vqa   = self.vqa_encoder(vqa_feat)
        fused = self.fusion(
            torch.cat([vis, vqa], dim=1)
        )
        if mode == 'reg':
            return [h(fused).squeeze(-1)
                    for h in self.heads_reg]
        return [h(fused) for h in self.heads_cls]

test_m = CDTHybridModelV2('vit').to(device)
d_img  = torch.randn(2, 3, 224, 224).to(device)
d_vqa  = torch.randn(2, VQA_DIM_V2).to(device)
out    = test_m(d_img, d_vqa)
print(f'Output: {[o.shape for o in out]}')
print(f'VQA encoder dim: {test_m.vqa_encoder.output_dim}')

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, pred, target):
        ce   = F.cross_entropy(
            pred, target, reduction='none'
        )
        pt   = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            loss = self.alpha[target] * loss
        return loss.mean()

class FocalLossReg(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, pred, target):
        mse  = F.mse_loss(
            pred, target, reduction='none'
        )
        err  = mse / 4.0
        loss = (err ** self.gamma) * mse
        return loss.mean()

def compute_alpha(train_df, domain_col, device):
    counts = train_df[domain_col].value_counts()
    total  = len(train_df)
    alpha  = torch.zeros(3)
    for cls in [0, 1, 2]:
        n          = counts.get(cls, 1)
        alpha[cls] = total / (3 * n)
    alpha = alpha / alpha.sum() * 3
    return alpha.to(device)

DOMAIN_WEIGHTS = torch.tensor(
    [1.0, 1.5, 1.0, 1.0, 1.0]
).to(device)

domain_cols   = [
    'domain_A','domain_B','domain_C',
    'domain_D','domain_E'
]
domain_alphas = []
for i, col in enumerate(domain_cols):
    alpha = compute_alpha(train_df, col, device)

    if col in ['domain_D', 'domain_E']:
        alpha[0] *= 3.0
        alpha[2] *= 2.0
        alpha = alpha / alpha.sum() * 3

    domain_alphas.append(alpha)
    print(f'{col}: {alpha.cpu().numpy().round(2)}')

def compute_focal_loss(outputs, labels,
                       mode='reg', gamma=2.0):
    total = 0

    gammas = {
        0: gamma,
        1: gamma,
        2: gamma,
        3: gamma * 0.3,
        4: gamma * 0.3,
    }

    for i, output in enumerate(outputs):
        g = gammas[i]
        if mode == 'cls':
            fl   = FocalLoss(
                gamma=g,
                alpha=domain_alphas[i]
            )
            loss = fl(output, labels[:,i].long())
        else:
            fl   = FocalLossReg(gamma=g)
            loss = fl(output, labels[:,i].float())
        total += loss * DOMAIN_WEIGHTS[i]
    return total / DOMAIN_WEIGHTS.sum()

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0,2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

def evaluate_hybrid(model, loader, mode='reg'):
    model.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                images, feat, labels = batch
                feat = feat.to(device)
            else:
                images, labels = batch
                feat = None
            images = images.to(device)
            if feat is not None:
                outputs = model(images, feat, mode)
            else:
                outputs = model(images, mode=mode)
            preds = get_predictions(outputs, mode)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    pred_total    = all_preds.sum(axis=1)
    true_total    = all_labels.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(true_dementia, -pred_total)
    except:
        auc = 0
    return {
        'sensitivity'  : sens,
        'specificity'  : spec,
        'auc'          : auc,
        'all_preds'    : all_preds,
        'all_labels'   : all_labels,
        'pred_total'   : pred_total,
        'true_dementia': true_dementia,
    }

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

TARGET_NORMAL   = 2000
TARGET_DEMENTIA = 1200

dementia_df    = train_df[train_df['dementia_risk']]
normal_df      = train_df[~train_df['dementia_risk']]
normal_sampled = normal_df.sample(
    n=min(TARGET_NORMAL, len(normal_df)),
    random_state=42
)

need_more = max(
    0, TARGET_DEMENTIA - len(dementia_df)
)
aug_rows  = []
generated = 0

if need_more > 0:
    while generated < need_more:
        for _, row in dementia_df.iterrows():
            if generated >= need_more:
                break
            try:
                src = f'{HUMAN_FOLDER}/{row["filename"]}'
                img = Image.open(src).convert('RGB')
                factor = random.uniform(0.8, 1.2)
                img    = ImageEnhance.Brightness(
                    img
                ).enhance(factor)
                new_name = (
                    f'aug_s_{generated:05d}_'
                    f'{row["filename"]}'
                )
                img.save(
                    f'{AUG_FOLDER}/{new_name}',
                    'JPEG', quality=95
                )
                orig_feat = get_vqa_feat(
                    row['filename'], vqa_v2
                )
                if orig_feat:
                    vqa_v2[new_name] = orig_feat
                new_row             = row.copy()
                new_row['filename'] = new_name
                aug_rows.append(new_row)
                generated += 1
            except:
                continue

    aug_df      = pd.DataFrame(aug_rows)
    dementia_df = pd.concat(
        [dementia_df, aug_df], ignore_index=True
    )

train_balanced = pd.concat(
    [normal_sampled, dementia_df],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(
    drop=True
)

train_ds = CDTHybridDatasetV2(
    train_balanced, HUMAN_FOLDER,
    vqa_v2, AUG_FOLDER, train_transform
)
val_ds   = CDTHybridDatasetV2(
    val_df, HUMAN_FOLDER,
    vqa_v2, None, val_transform
)
test_ds  = CDTHybridDatasetV2(
    test_df, HUMAN_FOLDER,
    vqa_v2, None, val_transform
)

train_loader = DataLoader(
    train_ds, batch_size=16,
    shuffle=True, num_workers=2
)
val_loader   = DataLoader(
    val_ds, batch_size=16,
    shuffle=False, num_workers=2
)
test_loader  = DataLoader(
    test_ds, batch_size=16,
    shuffle=False, num_workers=2
)

print(f'Train: {len(train_ds)} รูป')
print(f'Val  : {len(val_ds)} รูป')
print(f'Test : {len(test_ds)} รูป')

In [ ]:
def train_hybrid_v2(model_name, backbone='vit',
                    mode='reg', gamma=2.0,
                    epochs=15, patience=10):
    print(f'\n{"="*60}')
    print(f'Training: {model_name}')
    print(f'Backbone={backbone} Mode={mode} '
          f'Gamma={gamma}')
    print(f'VQA features v2 ({VQA_DIM_V2} dims)')
    print(f'Domain-Specific Encoder')
    print(f'Focal Loss gamma={gamma}')
    print(f'{"="*60}')

    model     = CDTHybridModelV2(backbone).to(device)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    best_combined = -1
    best_state    = None
    no_improve    = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, feat, labels in train_loader:
            images = images.to(device)
            feat   = feat.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images, feat, mode)
            loss    = compute_focal_loss(
                outputs, labels, mode, gamma
            )
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        val_results = evaluate_hybrid(
            model, val_loader, mode
        )

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, feat, labels in val_loader:
                images = images.to(device)
                feat   = feat.to(device)
                labels = labels.to(device)
                outputs = model(images, feat, mode)
                loss    = compute_focal_loss(
                    outputs, labels, mode, gamma
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        combined = (
            val_results['sensitivity'] +
            val_results['specificity']
        ) / 2

        if combined > best_combined:
            best_combined = combined
            best_state    = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stop epoch {epoch+1}')
                break

        print(
            f'Epoch {epoch+1:2d}/{epochs} | '
            f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
            f'Sens: {val_results["sensitivity"]:.3f} | '
            f'Spec: {val_results["specificity"]:.3f} | '
            f'AUC: {val_results["auc"]:.3f}'
        )

    model.load_state_dict(best_state)
    torch.save(
        best_state,
        f'{MODEL_FOLDER}/{model_name}_best.pth'
    )
    test_results = evaluate_hybrid(
        model, test_loader, mode
    )
    print(f'\nTest Results:')
    print(f'  Sens: {test_results["sensitivity"]:.3f}')
    print(f'  Spec: {test_results["specificity"]:.3f}')
    print(f'  AUC : {test_results["auc"]:.3f}')
    return model, test_results

In [ ]:
results_v2 = {}

# ลอง ViT REG + Focal
for gamma in [2.0]:
    model_name = f'hybrid_v2_vit_reg_g{int(gamma*10)}'
    path       = f'{MODEL_FOLDER}/{model_name}_best.pth'

    if os.path.exists(path):
        print(f'โหลด {model_name}')
        m = CDTHybridModelV2('vit').to(device)
        m.load_state_dict(
            torch.load(path, map_location=device)
        )
        m.eval()
        r = evaluate_hybrid(m, test_loader, 'reg')
    else:
        m, r = train_hybrid_v2(
            model_name, 'vit', 'reg', gamma
        )

    results_v2[model_name] = r

# ViT CLS + Focal
model_name = 'hybrid_v2_vit_cls_g20'
path       = f'{MODEL_FOLDER}/{model_name}_best.pth'

if os.path.exists(path):
    print(f'โหลด {model_name}')
    m = CDTHybridModelV2('vit').to(device)
    m.load_state_dict(
        torch.load(path, map_location=device)
    )
    m.eval()
    r = evaluate_hybrid(m, test_loader, 'cls')
else:
    m, r = train_hybrid_v2(
        model_name, 'vit', 'cls', 2.0
    )

results_v2[model_name] = r

In [ ]:
# เปลี่ยนชื่อ model ไม่ทับของเดิม
model_name = 'hybrid_v2_vit_cls_g20_fixed'
path       = f'{MODEL_FOLDER}/{model_name}_best.pth'

if os.path.exists(path):
    print(f'โหลด {model_name}')
    m = CDTHybridModelV2('vit').to(device)
    m.load_state_dict(
        torch.load(path, map_location=device)
    )
    m.eval()
    r = evaluate_hybrid(m, test_loader, 'cls')
else:
    m, r = train_hybrid_v2(
        model_name, 'vit', 'cls', gamma=2.0
    )

print(f'\n📊 Results:')
print(f'  Sens: {r["sensitivity"]:.3f}')
print(f'  Spec: {r["specificity"]:.3f}')
print(f'  AUC : {r["auc"]:.3f}')

# เปรียบกับเดิม
print(f'\nเปรียบกับ version เดิม:')
print(f'{"":25} {"Sens":>8} {"Spec":>8} {"AUC":>8}')
print('-'*50)
print(f'{"07s v2 cls (เดิม)":<25} '
      f'{0.911:>8.3f} {0.668:>8.3f} {0.892:>8.3f}')
print(f'{"07s v2 cls fixed":<25} '
      f'{r["sensitivity"]:>8.3f} '
      f'{r["specificity"]:>8.3f} '
      f'{r["auc"]:>8.3f}')

In [ ]:
all_results = {
    '07k ViT REG'       : {
        'sens':0.751,'spec':0.844,'auc':0.882
    },
    '07l Hybrid VQA'    : {
        'sens':0.760,'spec':0.850,'auc':0.887
    },
    '07n Downsample'    : {
        'sens':0.791,'spec':0.814,'auc':0.880
    },
    '07o Binary'        : {
        'sens':0.816,'spec':0.823,'auc':0.866
    },
}

for name, r in results_v2.items():
    all_results[f'07s {name}'] = {
        'sens': r['sensitivity'],
        'spec': r['specificity'],
        'auc' : r['auc'],
    }

print('='*75)
print('FINAL COMPARISON — All Experiments')
print('='*75)
print(f'{"Model":<30} {"Sens":>10} '
      f'{"Spec":>10} {"AUC":>10}')
print('='*75)

best_auc = max(
    v['auc'] for v in all_results.values()
)
for name, r in all_results.items():
    mark = '✅' if r['auc'] == best_auc else ''
    goal = '🎯' if (
        r['sens'] >= 0.88 and r['spec'] >= 0.82
    ) else ''
    print(
        f'{name:<30} '
        f'{r["sens"]:>10.3f} '
        f'{r["spec"]:>10.3f} '
        f'{r["auc"]:>10.3f} {mark}{goal}'
    )

print('='*75)
print(
    f'{"Target":<30} {"≥0.880":>10} '
    f'{"≥0.820":>10} {"≥0.910":>10}'
)
with open(
    f'{RESULT_FOLDER}/07s_final_results.json', 'w'
) as f:
    json.dump({
        k: {
            m: float(v)
            for m, v in r.items()
            if isinstance(v, (int, float))
        }
        for k, r in all_results.items()
    }, f, indent=2)

In [ ]:
model_key = 'hybrid_v2_vit_reg_g20'
best_model = CDTHybridModelV2('vit').to(device)
best_model.load_state_dict(torch.load(
    f'{MODEL_FOLDER}/{model_key}_best.pth',
    map_location=device
))
best_model.eval()

preds_list  = []
labels_list = []
raw_list    = []

with torch.no_grad():
    for images, feat, labels in test_loader:
        images  = images.to(device)
        feat    = feat.to(device)
        outputs = best_model(images, feat, 'reg')
        preds   = get_predictions(outputs, 'reg')
        raw     = torch.stack(
            [o.squeeze(-1) for o in outputs], dim=1
        )
        preds_list.append(preds.cpu())
        labels_list.append(labels.cpu())
        raw_list.append(raw.cpu())

preds_arr  = torch.cat(preds_list).numpy()
labels_arr = torch.cat(labels_list).numpy()
raw_arr    = torch.cat(raw_list).numpy()

domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for d_idx, d_name in domain_names.items():
    true_d = labels_arr[:, d_idx].astype(int)
    pred_d = preds_arr[:, d_idx].astype(int)

    cm      = confusion_matrix(
        true_d, pred_d, labels=[0,1,2]
    )
    cm_norm = cm.astype(float)
    rs      = cm.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1
    cm_norm = cm_norm / rs

    for row_idx, (cm_data, fmt, title_sfx) in enumerate([
        (cm, 'd', ''),
        (cm_norm, '.2f', '(normalized)')
    ]):
        ax = axes[row_idx][d_idx]
        sns.heatmap(
            cm_data, annot=True, fmt=fmt,
            cmap='Blues', ax=ax,
            xticklabels=['0','1','2'],
            yticklabels=['0','1','2'],
            cbar=False
        )
        acc = (true_d == pred_d).mean()
        ax.set_title(
            f'{d_name}\nAcc={acc:.3f} {title_sfx}',
            fontsize=8, fontweight='bold'
        )
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

plt.suptitle(
    f'Confusion Matrix — 07s {model_key}\n'
    f'Hybrid V2 + Focal Loss + Features v2',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07s_confusion_matrix.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

colors_true = {0:'#3F51B5', 1:'#00BCD4', 2:'#8BC34A'}

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for d_idx, d_name in domain_names.items():
    ax      = axes[d_idx]
    true_d  = labels_arr[:, d_idx].astype(int)
    score_d = raw_arr[:, d_idx]

    for true_val in [0, 1, 2]:
        mask   = (true_d == true_val)
        scores = score_d[mask]
        n      = mask.sum()
        if n < 2:
            continue
        try:
            kde    = gaussian_kde(scores, bw_method=0.3)
            x_vals = np.linspace(
                scores.min()-0.2,
                scores.max()+0.2, 300
            )
            y_vals = kde(x_vals)
            ax.plot(
                x_vals, y_vals,
                color=colors_true[true_val],
                linewidth=2,
                label=f'true={true_val} (n={n})'
            )
            ax.fill_between(
                x_vals, y_vals,
                color=colors_true[true_val],
                alpha=0.15
            )
        except:
            pass

    for x in [0.5, 1.5]:
        ax.axvline(x=x, color='gray',
                  linestyle='--', alpha=0.4)
    ax.set_title(d_name, fontsize=9,
                 fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Density')
    if d_idx == 0:
        ax.legend(fontsize=7)

plt.suptitle(
    'Score Distribution — 07s Hybrid V2\n'
    'blue=0  cyan=1  green=2',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07s_distribution.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
best_model = CDTHybridModelV2('vit').to(device)
best_model.load_state_dict(torch.load(
    f'{MODEL_FOLDER}/{model_name}_best.pth',
    map_location=device
))
best_model.eval()

preds_list  = []
labels_list = []
raw_list    = []

with torch.no_grad():
    for images, feat, labels in test_loader:
        images  = images.to(device)
        feat    = feat.to(device)
        outputs = best_model(images, feat, 'cls')
        preds   = get_predictions(outputs, 'cls')
        raw     = torch.stack([
            (torch.softmax(o, dim=1) *
             torch.tensor([0.,1.,2.]).to(device)
            ).sum(dim=1)
            for o in outputs
        ], dim=1)
        preds_list.append(preds.cpu())
        labels_list.append(labels.cpu())
        raw_list.append(raw.cpu())

preds_arr  = torch.cat(preds_list).numpy()
labels_arr = torch.cat(labels_list).numpy()
raw_arr    = torch.cat(raw_list).numpy()

In [ ]:
from sklearn.metrics import (
    confusion_matrix, roc_auc_score
)

pred_total    = preds_arr.sum(axis=1)
true_total    = labels_arr.sum(axis=1)
pred_dementia = (pred_total < CUTOFF).astype(int)
true_dementia = (true_total < CUTOFF).astype(int)

tn, fp, fn, tp = confusion_matrix(
    true_dementia, pred_dementia, labels=[0,1]
).ravel()
sens = tp/(tp+fn) if (tp+fn) > 0 else 0
spec = tn/(tn+fp) if (tn+fp) > 0 else 0
auc  = roc_auc_score(true_dementia, -pred_total)

domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}

print('='*65)
print(f'SUMMARY — {model_name}')
print('='*65)
print(f'\n📊 Overall:')
print(f'  Sensitivity: {sens:.3f} '
      f'{"✅" if sens>=0.88 else "⚠️"} (target ≥0.880)')
print(f'  Specificity: {spec:.3f} '
      f'{"✅" if spec>=0.82 else "⚠️"} (target ≥0.820)')
print(f'  AUC        : {auc:.3f} '
      f'{"✅" if auc>=0.91 else "⚠️"} (target ≥0.910)')

print(f'\n📊 Per-Domain Accuracy:')
print(f'{"Domain":<25} {"Acc":>8} '
      f'{"C0%":>8} {"C1%":>8} {"C2%":>8}')
print('-'*55)
for d_idx, d_name in domain_names.items():
    true_d = labels_arr[:, d_idx].astype(int)
    pred_d = preds_arr[:, d_idx].astype(int)
    acc    = (true_d == pred_d).mean()
    c_acc  = []
    for v in [0, 1, 2]:
        mask = (true_d == v)
        c_acc.append(
            (pred_d[mask] == v).mean() * 100
            if mask.sum() > 0 else 0
        )
    mark = '🟢' if acc >= 0.7 else \
           '🟡' if acc >= 0.5 else '🔴'
    print(f'{d_name:<25} {acc:>8.3f} '
          f'{c_acc[0]:>7.1f}% '
          f'{c_acc[1]:>7.1f}% '
          f'{c_acc[2]:>7.1f}% {mark}')

print(f'\n📊 Cutoff Analysis:')
print(f'{"Cutoff":>8} {"Sens":>8} '
      f'{"Spec":>8} {"F1":>8}')
print('-'*38)
from sklearn.metrics import f1_score
for cutoff in [4, 5, 6, 7, 8]:
    pred = (pred_total < cutoff).astype(int)
    try:
        tn2, fp2, fn2, tp2 = confusion_matrix(
            true_dementia, pred, labels=[0,1]
        ).ravel()
        s  = tp2/(tp2+fn2) if (tp2+fn2) > 0 else 0
        sp = tn2/(tn2+fp2) if (tn2+fp2) > 0 else 0
        f1 = f1_score(true_dementia, pred)
    except:
        s, sp, f1 = 0, 0, 0
    mark = '🎯' if s>=0.88 and sp>=0.82 else ''
    print(f'{cutoff:>8} {s:>8.3f} '
          f'{sp:>8.3f} {f1:>8.3f} {mark}')

all_r = {
    '07k ViT REG'     : (0.751, 0.844, 0.882),
    '07l Hybrid VQA'  : (0.760, 0.850, 0.887),
    '07n Downsample'  : (0.791, 0.814, 0.880),
    '07r Focal g=2.0' : (0.884, 0.701, 0.893),
    '07s v2 cls fixed': (sens,  spec,  auc),
}
print(f'{"Model":<25} {"Sens":>8} '
      f'{"Spec":>8} {"AUC":>8}')
print('='*50)
best_auc = max(v[2] for v in all_r.values())
for name, (s, sp, a) in all_r.items():
    mark = '✅' if a == best_auc else ''
    goal = '🎯' if s>=0.88 and sp>=0.82 else ''
    print(f'{name:<25} {s:>8.3f} '
          f'{sp:>8.3f} {a:>8.3f} {mark}{goal}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for d_idx, d_name in domain_names.items():
    true_d = labels_arr[:, d_idx].astype(int)
    pred_d = preds_arr[:, d_idx].astype(int)

    cm      = confusion_matrix(
        true_d, pred_d, labels=[0,1,2]
    )
    cm_norm = cm.astype(float)
    rs      = cm.sum(axis=1, keepdims=True)
    rs[rs==0] = 1
    cm_norm = cm_norm / rs

    for row_idx, (data, fmt, sfx) in enumerate([
        (cm,      'd',   ''),
        (cm_norm, '.2f', '(normalized)')
    ]):
        ax = axes[row_idx][d_idx]
        sns.heatmap(
            data, annot=True, fmt=fmt,
            cmap='Blues', ax=ax,
            xticklabels=['0','1','2'],
            yticklabels=['0','1','2'],
            cbar=False
        )
        acc = (true_d == pred_d).mean()
        ax.set_title(
            f'{d_name}\nAcc={acc:.3f} {sfx}',
            fontsize=8, fontweight='bold'
        )
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

plt.suptitle(
    f'Confusion Matrix — {model_name}\n'
    f'Sens={sens:.3f} Spec={spec:.3f} AUC={auc:.3f}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/{model_name}_confusion.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
colors_true = {0:'#3F51B5', 1:'#00BCD4', 2:'#8BC34A'}

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for d_idx, d_name in domain_names.items():
    ax      = axes[d_idx]
    true_d  = labels_arr[:, d_idx].astype(int)
    score_d = raw_arr[:, d_idx]

    for true_val in [0, 1, 2]:
        mask   = (true_d == true_val)
        scores = score_d[mask]
        n      = mask.sum()
        if n < 2:
            continue
        try:
            from scipy.stats import gaussian_kde
            kde    = gaussian_kde(scores, bw_method=0.3)
            x_vals = np.linspace(
                scores.min()-0.2,
                scores.max()+0.2, 300
            )
            ax.plot(
                x_vals, kde(x_vals),
                color=colors_true[true_val],
                linewidth=2,
                label=f'true={true_val} (n={n})'
            )
            ax.fill_between(
                x_vals, kde(x_vals),
                color=colors_true[true_val],
                alpha=0.15
            )
        except:
            pass

    for x in [0.5, 1.5]:
        ax.axvline(x=x, color='gray',
                  linestyle='--', alpha=0.4)
    ax.set_title(d_name, fontsize=9,
                 fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Density')
    if d_idx == 0:
        ax.legend(fontsize=7)

plt.suptitle(
    f'Distribution — {model_name}\n'
    f'blue=0  cyan=1  green=2',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/{model_name}_distribution.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

all_results_plot = {
    '07k ViT REG'     : {
        'sens':0.751,'spec':0.844,'auc':0.882,
        'color':'#9E9E9E'
    },
    '07l Hybrid VQA'  : {
        'sens':0.760,'spec':0.850,'auc':0.887,
        'color':'#2196F3'
    },
    '07n Downsample'  : {
        'sens':0.791,'spec':0.814,'auc':0.880,
        'color':'#4CAF50'
    },
    '07r Focal g=2.0' : {
        'sens':0.884,'spec':0.701,'auc':0.893,
        'color':'#FF9800'
    },
    '07s v2 fixed'    : {
        'sens':sens,'spec':spec,'auc':auc,
        'color':'#F44336'
    },
}

names  = list(all_results_plot.keys())
colors = [v['color'] for v in
          all_results_plot.values()]

for ax_idx, (metric, title, target) in enumerate([
    ('sens', 'Sensitivity', 0.88),
    ('spec', 'Specificity', 0.82),
    ('auc',  'AUC-ROC',     0.91),
]):
    vals = [all_results_plot[n][metric]
            for n in names]
    bars = axes[ax_idx].barh(
        names, vals, color=colors,
        edgecolor='white', height=0.6
    )
    axes[ax_idx].axvline(
        x=target, color='red',
        linestyle='--', linewidth=1.5,
        label=f'Target {target}'
    )
    axes[ax_idx].set_title(
        title, fontsize=12, fontweight='bold'
    )
    axes[ax_idx].set_xlim(0.5, 1.05)
    axes[ax_idx].legend(fontsize=8)
    for bar, val in zip(bars, vals):
        axes[ax_idx].text(
            val + 0.003,
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}',
            va='center', fontsize=8
        )

plt.suptitle(
    'Model Comparison — All Experiments',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/final_comparison.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

with open(
    f'{RESULT_FOLDER}/{model_name}_results.json',
    'w'
) as f:
    json.dump({
        'model'      : model_name,
        'sensitivity': float(sens),
        'specificity': float(spec),
        'auc'        : float(auc),
    }, f, indent=2)